# Method 2 — Full pipeline & đánh giá (Kaggle T4)

Phase 5 của `docs/method2_plan.md`: chạy oracle + full pipeline, xuất predictions theo contract, đo latency 4 giai đoạn. Ngân sách ~1h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 1: env =====
!pip install -q 'sentence-transformers>=3.0' peft transformers accelerate \
                jsonschema rank_bm25 datasets

import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
# Kỳ vọng: Tesla T4, ~15.0 GB free. T4 KHÔNG có bf16 → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: mount code + data =====
# Đẩy repo và data lên Kaggle Dataset (private) trước.
!cp -r /kaggle/input/toolcalling-vi-src/src /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-src/configs /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-data/data /kaggle/working/data
%cd /kaggle/working

import json, os, glob, sys
sys.path.insert(0, '/kaggle/working')

# Cache model HF thành Kaggle Dataset để không tải lại mỗi session.
os.environ.setdefault('HF_HOME', '/kaggle/input/hf-cache')

manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('snapshot commit:', manifest.get('git_commit'))


## Hai chế độ bắt buộc (§6.3 experimental_plan)

| Chế độ | Input Cross-Encoder | Trả lời câu hỏi |
|---|---|---|
| `oracle` | Tool gold | Extraction tốt đến đâu, độc lập retrieval |
| `pipeline` | Bi-Encoder top-k + abstention | Hiệu năng hệ thống thật |

`ArgA_oracle − ArgA_pipeline` = phần lỗi do retrieval. Con số này phải xuất
hiện tường minh trong báo cáo.


In [ ]:
for gold, tag in [('data/custom_vi/v1/test_seen.jsonl', 'custom_seen'),
                  ('data/custom_vi/v1/test_unseen.jsonl', 'custom_unseen'),
                  ('data/benchmark_vi/test.jsonl', 'benchmark')]:
    for mode in ['pipeline', 'oracle']:
        !python -m src.models.pipeline.method2 \
            --config configs/method2/pipeline.yaml \
            --gold {gold} --mode {mode} \
            --output-dir results/method2/predictions/{tag}


## Chạy evaluator chung

Cùng normalization, cùng rule so khớp với ba method còn lại — xem
`docs/evaluation.md`.


In [ ]:
!python -m src.evaluation.cli evaluate \
    --gold data/custom_vi/v1/test_seen.jsonl \
    --predictions results/method2/predictions/custom_seen/predictions.jsonl \
    --oracle-predictions results/method2/predictions/custom_seen/oracle_predictions.jsonl \
    --slice metadata.tool_split \
    --output-dir results/evaluation/method_2_seen


## Latency tách 4 giai đoạn — `t_index_build` ghi riêng, không cộng vào


In [ ]:
latency = json.load(open('results/method2/predictions/custom_seen/latency_pipeline.json', encoding='utf-8'))
for stage in ['t_query_embed', 't_retrieve', 't_cross_encode', 't_validate', 'total']:
    print(f"{stage:16s} p50={latency[stage]['p50_ms']:8.2f} ms  p95={latency[stage]['p95_ms']:8.2f} ms")

index_meta = json.load(open('data/method2/index/index_meta.json', encoding='utf-8'))
print('\nt_index_build (một lần, KHÔNG cộng vào latency/query):', index_meta['t_index_build_sec'], 's')


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/eval_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/eval_run.tar.gz
